# Qt–Ft agglomeration: control panel

1. Set **all** parameters in the cell below (LAMMPS `units nano`: nm, ns, zJ, pN, ag).
2. Run it: the derived values (γ, stability α, binding probability, …) and checks are printed.
3. Build the run folder, start the simulation in the background, follow its progress.

The run folder (`qtft-data/single/<run name>/` for one replica, `qtft-data/ensemble/<run name>/` for
several, inside the repository but not in git) holds `params.json` and one `rep_XX/in.lammps` per replica;
it is the record of what was run, not this notebook.

In [ ]:
%matplotlib inline
from qtft import Params, build_run, start, stop, progress, watch, quicklook

## 1. Parameters

In [ ]:
p = Params(
    # --- system (prototype: 400/1600 in 928.3 nm; full system: 4000/16000 in 2000 nm)
    n_qt=4000,                 # number of Qt encapsulins
    n_ft=10000,                # number of Ft ferritins
    box_length=2000.0,         # cubic periodic box edge [nm]

    # --- particles
    radius_qt=27.5,            # effective radius [nm]
    radius_ft=13.0,            # effective radius [nm]
    D_qt=2e-4,                 # diffusion coefficient [nm^2/ns]  -> gamma = kT/D
    D_ft=5e-4,                 # diffusion coefficient [nm^2/ns]
    temperature=310.0,         # [K]

    # --- soft repulsion E = k (r_c - r)^2, r_c = r_i + r_j   [zJ/nm^2]
    k_qq=3.32,
    k_ff=2.0,
    k_qf=1.245,

    # --- harmonic bond E = K (r - r0)^2
    bond_k=1.66,               # [zJ/nm^2]
    bond_r0=None,              # [nm]; None -> r_Qt + r_Ft

    # --- binding (fix bond/create)
    k_on=1e-6,                 # microscopic rate per Qt-Ft pair within R_bind [1/ns]
    bind_radius=None,          # [nm]; None -> r_Qt + r_Ft
    bind_every=1,              # attempt every N steps
    maxbond_qt=60,             # max bonds per Qt
    maxbond_ft=10,             # max bonds per Ft

    # --- run
    timestep=500.0,            # [ns]
    equil_steps=5000,          # equilibration, binding off
    prod_steps=1_000_000,      # production, binding on (1_000_000 x 500 ns = 500 ms)

    # --- output / in-run analysis
    output_every=1000,         # thermo + analysis, bonds.dump, rdf, histograms [steps]
    traj_every=1000,           # traj.dump [steps]
    large_cluster_min=5,
    rdf_bins=200,
    rdf_cutoff=None,           # [nm]; None -> 3 (r_Qt + r_Ft)

    # --- technical
    neighbor_skin=20.0,        # [nm]; benchmark: 20 is 20-45 % faster than 10
    extra_special=1000,
    comm_cutoff=None,          # [nm]; None -> automatic
    insert_maxtry=1000,

    # --- execution
    n_replicas=1,
    mpi_procs=4,               # MPI processes per replica (benchmark: 4 for ensembles)
    max_parallel=4,            # replicas at once; None -> cores // mpi_procs
    master_seed=12345,
    data_dir="qtft-data",        # relative -> inside the repository; runs go to single/ or ensemble/
    run_name=None,             # None -> generated from the parameters
    lmp_command="lmp",
    mpirun_command="mpirun --bind-to none",  # no core pinning
)

print("run folder:", p.run_dir)
print(p.summary())

## 2. Build the run folder

In [ ]:
run_dir = build_run(p)          # build_run(p, overwrite=True) replaces an existing folder
print(run_dir)

In [ ]:
# optional: look at the generated LAMMPS input of replica 1
print((run_dir / "rep_01" / "in.lammps").read_text())

## 3. Run (in the background — closing the notebook does not stop it)

`watch(run_dir)` shows the progress live (refreshed every 10 s) until the run has finished; the stop
button only ends the view, not the run. The same information goes to `launcher.log` every minute:
follow it in an Ubuntu terminal with `tail -f <run_dir>/launcher.log`.

In [ ]:
start(run_dir)

In [ ]:
watch(run_dir)                  # live view; progress(run_dir) gives a one-off table

In [ ]:
# stop(run_dir)                 # uncomment to stop all replicas of this run

## 4. Quick look

In [ ]:
quicklook(run_dir);

To continue with an existing run after restarting the kernel:

```python
from qtft import data_path
run_dir = data_path() / "ensemble" / "<run name>"   # or "single"
```

## 5. Performance measurement (once per machine)

Runs short productions of the system defined in `p` with 1, 2, 4, … MPI processes per replica (up to the
physical cores), each time filling the machine with replicas. This is done twice: filling all hardware
threads and filling only the physical cores, to see whether hyperthreading helps. Takes a few minutes for
the full system. Use the recommendation to set `mpi_procs` and `max_parallel` above.
Results: `qtft-data/benchmark/<cores>cores/results.csv`.

In [ ]:
import os
import pandas as pd
from qtft.benchmark import benchmark, physical_cores, recommend

bench = pd.concat([
    benchmark(
        p,                                   # system and potentials as set above
        cores=cores,                         # hardware threads to fill with replicas
        steps=2000,                          # production steps per measurement
        variants={"skin 10": {"neighbor_skin": 10.0}, "skin 20": {"neighbor_skin": 20.0}},
    )
    for cores in sorted({os.cpu_count(), physical_cores()}, reverse=True)
], ignore_index=True)
bench.round(2)

In [ ]:
recommend(bench, n_replicas=p.n_replicas)

After a long run: how many 1-2/1-3/1-4 neighbours did atoms really need? `extra_special`
(default 1000) must stay above the maximum.

In [ ]:
from qtft.benchmark import special_need
special_need(run_dir / "rep_01")